# Vessel card — evidence and preventive action

Notebook 02 produces `V001, 85.5, High, [Repetition, Trend, Unresolved, Maintenance]`. Nobody can
act on that. A superintendent reads it and asks what to actually do, and the score has no answer.

This notebook closes that gap. It takes each vessel's drivers and attaches:

- **the source records behind them** — which deficiency IDs, which audit finding, which overdue job
- **a recommended action, an owning department and a target window**


The output is the eight-column view the brief asks for in Section 6:

> Current Risk → Why → Evidence → Critical Issues → Action → Ownership → Timing → Readiness

### 0. Setup

In [12]:
import sys
from pathlib import Path

for _dir in (Path.cwd(), *Path.cwd().parents):
    if (_dir / 'pyproject.toml').exists():
        sys.path.insert(0, str(_dir))
        break

import pandas as pd

from feature_builder import build_features, REFERENCE_DATE, PROJECT_ROOT
from scoring import score_from_features, DIMENSIONS
from vessel_card import ACTIONS, evidence_for, since_last_inspection, recommendations_for

pd.set_option('display.width', 200)

features = build_features()
results = score_from_features(features)

print(f'{len(results)} vessels scored as at {REFERENCE_DATE.date()}')
print(results.tier.value_counts().reindex(['High', 'Medium', 'Low']).to_string())

20 vessels scored as at 2026-08-24
tier
High       2
Medium     2
Low       16


## 1. The evidence layer

The score says a vessel is high risk. This layer says *on what basis* — by going back to the
original records and naming them.

Each driver gets its own lookup, and each returns the actual record IDs, not just a sentence.
That matters: a superintendent can open DEF0001, read the finding, and decide the system has it
wrong. Evidence nobody can check is just an opinion.

| Driver | What it goes and finds |
|---|---|
| Repetition | the defect that keeps coming back, how often, and which deficiency records |
| Trend | how many findings at each inspection, oldest to newest |
| Unresolved | what is still open, what was already open before the last inspection, and what the company had already flagged internally |
| Maintenance | which jobs are overdue and how long the oldest has been sitting |
| Concentration | which category the findings cluster in, and what share |
| Crew | who joined recently and the average experience on board |
| Equipment | which items have failed more than once |

In [4]:
v = 'V001'
row = results.loc[v]

print(f'{v} — score {row.score}, tier {row.tier}, drivers {row.drivers}\n')
for driver in row.drivers:
    ev = evidence_for(v, driver)
    print(f'  {driver}')
    print(f'    {ev["summary"]}')
    print(f'    records: {ev["record_ids"]}\n')

V001 — score 85.5, tier High, drivers ['Repetition', 'Trend', 'Unresolved', 'Maintenance']

  Repetition
    "Fire pump pressure low" recorded 9 times across 4 inspections
    records: ['DEF0001', 'DEF0002', 'DEF0003', 'DEF0005', 'DEF0006']

  Trend
    3 finding(s) at 2024-02-15 rising to 10 at 2025-06-29
    records: ['INS0001', 'INS0002', 'INS0003', 'INS0004']

  Unresolved
    10 deficiencies still open, 3 of them raised before the most recent inspection; 4 internal audit findings unresolved, 1 matching an open deficiency
    records: ['DEF0010', 'DEF0011', 'DEF0012', 'DEF0019', 'DEF0020', 'AUD0001', 'AUD0002', 'AUD0003', 'AUD0004']

  Maintenance
    11 planned jobs overdue, oldest due 2026-04-18 (128 days ago)
    records: ['MAIN0001', 'MAIN0002', 'MAIN0003', 'MAIN0004', 'MAIN0005']



The Unresolved line is the one worth reading twice. Three of V001's ten open deficiencies were
raised **before its most recent inspection** — meaning an inspector had already seen and recorded
them once, they were not closed, and they were still outstanding when the next inspector boarded.

## 2. Critical issues — what has changed since the last inspection

Every vessel in the fleet was last inspected **415 to 455 days ago**. So everything the score
is built from describes each ship as it was well over a year back. A lot has happened since:
equipment has failed, jobs have gone overdue, findings were never closed.

None of that was there for the last inspector to see. All of it will be there for the next one.

That gap is the whole reason an early-warning system exists — and it is filled entirely by the
features that showed no measurable link to past deficiency counts. They score almost nothing and
they carry most of what a superintendent actually needs to know today.

In [6]:
v = 'V004'

sli = since_last_inspection(v)
print(f'{v} — last inspected {sli["last_inspection"]}, {sli["days_since"]} days ago\n')
for k in ['equipment_failures', 'deficiencies_still_open', 'overdue_maintenance',
          'unresolved_audit_findings', 'crew_joined']:
    print(f'  {k.replace("_", " "):30} {sli[k]}')
print(f'\n  repeat offenders: {sli["equipment_detail"]}')

V004 — last inspected 2025-05-26, 455 days ago

  equipment failures             3
  deficiencies still open        5
  overdue maintenance            7
  unresolved audit findings      5
  crew joined                    6

  repeat offenders: {'ECDIS': 3}


In [7]:
# fleet-wide, to show the window is never empty
fleet = pd.DataFrame({vid: since_last_inspection(vid) for vid in results.index}).T
totals = fleet[['equipment_failures', 'deficiencies_still_open',
                'overdue_maintenance', 'unresolved_audit_findings']].sum()
print('accumulated across the fleet since each vessel was last inspected:')
print(totals.to_string())
print(f'\ndays since last inspection: {fleet.days_since.min()} to {fleet.days_since.max()}')

accumulated across the fleet since each vessel was last inspected:
equipment_failures           130
deficiencies_still_open       47
overdue_maintenance          121
unresolved_audit_findings     78

days since last inspection: 415 to 455


## 3. Preventive action

Nobody acts on a score. They act on a task with their name against it and a date.

So each driver becomes exactly that: what to do, which department does it, and by when. The
departments are the ones named in the brief — Marine, Technical, Crewing, Operations, Vessel.

Tasks are listed biggest contributor first, so task 1 is whatever pushed the score up most. How
urgent they are comes from the tier, not the driver — the same overdue maintenance backlog is
immediate on a High vessel and routine work on a Low one.

In [8]:
print(pd.DataFrame(ACTIONS).T[['owner', 'window']].to_string())

                            owner                   window
Repetition              Technical    Before next port call
Trend                      Marine                  30 days
Unresolved     Vessel / Technical                  14 days
Maintenance             Technical                  30 days
Concentration              Marine                  30 days
Crew                      Crewing  Before next crew change
Equipment               Technical                  30 days


## 4. The vessel card

All eight output areas assembled into one view. 

**Readiness** is the one field that is not a lookup. It answers "what would this vessel look like
if the recommended actions were completed?" — computed by clearing the closable items and
re-scoring. 

It is shown as a projection, not a promise, and the gap between current and projected
is itself informative: if closing everything actionable barely moves the score, the problem is
structural rather than a backlog.

In [9]:
from scoring import normalise, prepare, score_fleet, rank_fleet, SCORING_INPUTS

CLOSABLE = ['open_deficiencies', 'overdue_maintenance', 'open_audit', 'overdue_audit',
            'known_issues_unresolved', 'known_issue_occurrences']

# min-max bounds from the fleet as it stands today, held fixed for readiness
_prepared = prepare(features)
_BOUNDS = {c: (_prepared[c].min(), _prepared[c].max())
           for c in SCORING_INPUTS if c in _prepared.columns}


def readiness(vessel_id, features=features):
    '''Re-score one vessel with its actionable items cleared, on today's fleet scale.

    Only genuinely closable things are zeroed. History cannot be closed: repetition, trend
    and concentration describe inspections that already happened and stay as they are.

    The bounds are deliberately frozen. Re-normalising after the edit would rescale the
    whole fleet - removing V001's extreme values lifts every other vessel's score, and V004
    would appear to jump from 60.6 to 72.5 without anything changing aboard it. That is a
    true statement about a fleet-relative score and a useless answer to the question the
    card is asking, which is what THIS vessel would look like on today's scale.
    '''
    f = features.copy()
    f.loc[vessel_id, CLOSABLE] = 0
    p = prepare(f)

    norm = pd.DataFrame({
        c: ((p[c] - lo) / (hi - lo)).clip(0, 1) if hi > lo else pd.Series(0.0, index=p.index)
        for c, (lo, hi) in _BOUNDS.items()})

    scored = score_fleet(norm)
    return rank_fleet(scored).loc[vessel_id, ['score', 'tier']]


def vessel_card(vessel_id, results=results):
    r = results.loc[vessel_id]
    p = features.loc[vessel_id]
    proj = readiness(vessel_id)
    sli = since_last_inspection(vessel_id)

    print('=' * 78)
    print(f'{vessel_id}  {p.vessel_name}   {p.vessel_type}, {p.flag_state}, built {p.build_year}')
    print('=' * 78)

    print(f'\nCURRENT RISK   {r.score} / 100   tier {r.tier}   rank {r["rank"]} of {len(results)}')

    if not r.drivers:
        print('\nWHY            No dimension above the fleet 75th percentile. This vessel is not')
        print('               an outlier on any measure - see Critical Issues for current state.')
    else:
        print('\nWHY            ' + ', '.join(r.drivers))
        if r.saturated:
            print('               at maximum on: ' + ', '.join(r.saturated))

    print('\nEVIDENCE')
    # with no elevated driver, fall back to this vessel's own two largest dimensions so the
    # evidence still describes the vessel rather than an arbitrary pair
    fallback = r[list(DIMENSIONS)].astype(float).nlargest(2).index.tolist()
    for driver in (r.drivers or fallback):
        ev = evidence_for(vessel_id, driver)
        if ev:
            print(f'   {driver:14} {ev["summary"]}')
            print(f'   {"":14} records: {ev["record_ids"][:4]}')

    print(f'\nCRITICAL ISSUES   since last inspection {sli["last_inspection"]} '
          f'({sli["days_since"]} days)')
    print(f'   {sli["equipment_failures"]} equipment failures, '
          f'{sli["deficiencies_still_open"]} deficiencies still open, '
          f'{sli["overdue_maintenance"]} overdue jobs, '
          f'{sli["unresolved_audit_findings"]} unresolved audit findings')

    recs = recommendations_for(r.drivers, r.tier)
    if recs:
        print('\nACTION / OWNERSHIP / TIMING')
        for x in recs:
            print(f'   {x["priority"]}. {x["driver"]:14} {x["owner"]:20} {x["window"]:26} '
                  f'[{x["urgency"]}]')
            print(f'      {x["action"]}')
    else:
        print('\nACTION / OWNERSHIP / TIMING')
        print('   No driver-led action. Clear the overdue maintenance backlog and close')
        print('   outstanding audit findings as routine work.  Technical / Vessel, 30 days.')

    print(f'\nREADINESS      {r.score} now  ->  {proj.score} if the closable items above are '
          f'cleared  (tier {proj.tier})')
    print('=' * 78)


vessel_card('V004')

V004  MV Delta   Chemical Tanker, Marshall Islands, built 2011

CURRENT RISK   60.6 / 100   tier High   rank 2 of 20

WHY            Concentration, Trend, Unresolved, Repetition
               at maximum on: Concentration

EVIDENCE
   Concentration  13 of 18 findings in Navigation (72% of all findings)
                  records: ['DEF0057', 'DEF0058', 'DEF0059', 'DEF0060']
   Trend          2 finding(s) at 2024-02-15 rising to 7 at 2025-05-26
                  records: ['INS0013', 'INS0014', 'INS0015', 'INS0016']
   Unresolved     5 deficiencies still open, 0 of them raised before the most recent inspection; 5 internal audit findings unresolved, 1 matching an open deficiency
                  records: ['DEF0068', 'DEF0070', 'DEF0071', 'DEF0072']
   Repetition     "Chart correction records incomplete" recorded 4 times across 3 inspections
                  records: ['DEF0057', 'DEF0059', 'DEF0060', 'DEF0068']

CRITICAL ISSUES   since last inspection 2025-05-26 (455 days)
   3 equipment 

In [10]:
vessel_card('V016')

V016  MV Voyager   Bulk Carrier, Panama, built 2023

CURRENT RISK   25.1 / 100   tier Low   rank 5 of 20

WHY            No dimension above the fleet 75th percentile. This vessel is not
               an outlier on any measure - see Critical Issues for current state.

EVIDENCE
   Maintenance    7 planned jobs overdue, oldest due 2026-02-08 (197 days ago)
                  records: ['MAIN0306', 'MAIN0312', 'MAIN0313', 'MAIN0314']
   Trend          1 finding(s) at 2024-02-02 rising to 3 at 2025-06-13
                  records: ['INS0061', 'INS0062', 'INS0063', 'INS0064']

CRITICAL ISSUES   since last inspection 2025-06-13 (437 days)
   6 equipment failures, 1 deficiencies still open, 7 overdue jobs, 4 unresolved audit findings

ACTION / OWNERSHIP / TIMING
   No driver-led action. Clear the overdue maintenance backlog and close
   outstanding audit findings as routine work.  Technical / Vessel, 30 days.

READINESS      25.1 now  ->  14.0 if the closable items above are cleared  (tier Low)

## 5. Fleet view

The same information for every vessel, as an operations user would first see it.

In [11]:
fleet_view = pd.DataFrame({
    'vessel': features.vessel_name,
    'score': results.score,
    'tier': results.tier,
    'why': results.drivers.apply(lambda d: ', '.join(d) if d else 'no elevated dimension'),
    'first_action': results.drivers.apply(
        lambda d: ACTIONS[d[0]]['owner'] + ' / ' + ACTIONS[d[0]]['window'] if d else 'routine'),
}).sort_values('score', ascending=False)

print(fleet_view.to_string())

               vessel  score    tier                                           why                       first_action
vessel_id                                                                                                            
V001         MV Alpha   85.5    High    Repetition, Trend, Unresolved, Maintenance  Technical / Before next port call
V004         MV Delta   60.6    High  Concentration, Trend, Unresolved, Repetition                   Marine / 30 days
V019         MV Atlas   32.2  Medium      Maintenance, Equipment, Repetition, Crew                Technical / 30 days
V007       MV Horizon   29.4  Medium            Maintenance, Repetition, Equipment                Technical / 30 days
V013         MV Titan   25.1     Low                        Unresolved, Repetition       Vessel / Technical / 14 days
V016       MV Voyager   25.1     Low                         no elevated dimension                            routine
V014         MV Unity   25.0     Low                    

In [13]:
fleet_view.to_csv(PROJECT_ROOT / "generated_csvs" / 'fleet_view.csv')